## 1. Install

In [1]:
%%capture
# Kaggle's image already ships a working CUDA torch — reinstalling it is slow and
# frequently breaks the T4 image, so we install everything else around it.
!uv pip install -q --no-deps unsloth unsloth_zoo cut_cross_entropy
!uv pip install -q --no-deps "trl==0.22.2"
!uv pip install -q "transformers==4.56.2" "datasets>=3.0" "accelerate>=1.0" \
                "peft>=0.15" bitsandbytes "torchao>=0.16.0" hf_transfer msgspec tyro

In [2]:
import torch
print("torch      :", torch.__version__)
print("CUDA       :", torch.version.cuda)
print("GPU count  :", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i}  {p.name}  {p.total_memory/1e9:.1f} GB  sm_{p.major}{p.minor}")
assert torch.cuda.device_count() >= 2, \
    "Only 1 GPU visible — set Accelerator to 'GPU T4 x2' in Notebook settings."
print("bf16 supported:", torch.cuda.is_bf16_supported(), "(False on T4 -> we train in fp16)")

torch      : 2.10.0+cu128
CUDA       : 12.8
GPU count  : 2
  cuda:0  Tesla T4  15.6 GB  sm_75
  cuda:1  Tesla T4  15.6 GB  sm_75
bf16 supported: True (False on T4 -> we train in fp16)


## 2. Locate the dataset

Works either way: if the dataset is attached to the notebook it is found under
`/kaggle/input`; otherwise it is pulled with `kagglehub`.

In [3]:
import os, glob

# If kagglehub is needed, put your dataset slug here ("<username>/<dataset-slug>").
KAGGLE_DATASET = "YOUR_USERNAME/networkslt"

def looks_like_root(p):
    return all(os.path.isfile(f"{p}/{d}/{d}.jsonl") for d in ("train", "val", "test"))

def find_data_root():
    # 1) attached as a Kaggle dataset
    for cand in glob.glob("/kaggle/input/*/**/train", recursive=True):
        root = os.path.dirname(cand)
        if looks_like_root(root):
            return root
    # 2) fall back to kagglehub
    import kagglehub
    base = kagglehub.dataset_download(KAGGLE_DATASET)
    if looks_like_root(base):
        return base
    for cand in glob.glob(f"{base}/**/train", recursive=True):
        root = os.path.dirname(cand)
        if looks_like_root(root):
            return root
    raise FileNotFoundError(f"No train/val/test jsonl found under {base}")

DATA_ROOT = find_data_root()
print("DATA_ROOT =", DATA_ROOT)
for d in ("train", "val", "test"):
    f = f"{DATA_ROOT}/{d}/{d}.jsonl"
    print(f"  {d:5s} {sum(1 for _ in open(f)):>5,} pairs   {f}")

DATA_ROOT = /kaggle/input/datasets/manith04/networkslt
  train 2,416 pairs   /kaggle/input/datasets/manith04/networkslt/train/train.jsonl
  val     303 pairs   /kaggle/input/datasets/manith04/networkslt/val/val.jsonl
  test    302 pairs   /kaggle/input/datasets/manith04/networkslt/test/test.jsonl


## 3. Config

Everything tunable lives here; the training script reads it back from JSON so the notebook
stays the single source of truth.

In [4]:
import json, torch

WORK = "/kaggle/working"

CFG = {
    # --- model ---------------------------------------------------------------
    "model_name":        "unsloth/llama-3-8b-bnb-4bit",
    "load_in_4bit":      True,
    # Truncation ceilings only: trl's ORPO collator pads each batch to its own
    # longest item (~170 tokens here), so raising these costs no memory or time.
    # Longest example in the whole dataset: ~168 tokens prompt+completion.
    "max_seq_length":    1024,
    "max_prompt_length": 256,

    # --- LoRA ----------------------------------------------------------------
    "lora_r":     16,
    "lora_alpha": 16,
    # 0 is Unsloth's optimised fast path; 0.05-0.1 regularises instead, which
    # matters with 2,416 examples and up to 30 epochs. Set 0 back for max speed.
    "lora_dropout": 0,

    # --- ORPO ----------------------------------------------------------------
    "beta": 0.1,                # ORPO lambda: strength of the odds-ratio penalty

    # --- schedule ------------------------------------------------------------
    # effective batch = per_device x grad_accum x GPUs = 8 x 1 x 2 = 16 (unchanged)
    # Your sequences are ~170 tokens, so bs=2 left the T4s mostly idle and paid
    # the accumulation overhead 4x per step. Bigger micro-batch, no accumulation,
    # same effective batch = the same training run, fewer and fuller kernels.
    # Drop to bs=4 / accum=2 if you hit CUDA OOM.
    "per_device_train_batch_size": 8,
    "per_device_eval_batch_size":  16,
    "gradient_accumulation_steps": 1,

    # "unsloth" recomputes activations to save VRAM — essential at long context,
    # pure overhead at ~170 tokens. False trades a little VRAM for ~30% speed.
    "use_gradient_checkpointing": False,

    # Fixed, not a ratio: warmup_ratio scales off num_train_epochs, so with 30
    # epochs as an upper bound it warmed up for 3 full epochs before reaching LR.
    "warmup_steps": 100,
    # Upper bound, not a promise: early stopping ends the run once the val metric
    # stops improving, and the best epoch is what gets saved.
    "num_train_epochs": 30,
    "early_stopping_patience": 3,   # evals without improvement before stopping; 0 = off
    "learning_rate":    2e-5,   # ORPO+LoRA usually lands in 8e-6 .. 5e-5
    "logging_steps":    5,
    "seed":             3407,

    # --- when to evaluate and checkpoint -------------------------------------
    # "epoch" -> evaluate + checkpoint once per epoch. One candidate per epoch,
    #            and early-stopping patience is counted in epochs.
    # "steps" -> every eval_steps optimiser steps: finer selection, catches
    #            overfitting that starts mid-epoch, patience counted in evals.
    "eval_save_strategy": "epoch",
    "eval_steps":         50,    # only used when eval_save_strategy == "steps"
    "save_total_limit":   3,

    # --- model selection -----------------------------------------------------
    # The best checkpoint by this metric is reloaded at the end of training and
    # is what gets saved as the final adapter.
    "metric_for_best_model": "eval_loss",            # smooth, standard choice
    "greater_is_better":     False,
    # Alternative — select on preference separation on the val set instead, which
    # is the same quantity the final test cell reports. Coarser (steps of 1/303),
    # so it ties more often, and early stopping then triggers on ties:
    #   "metric_for_best_model": "eval_rewards/accuracies",
    #   "greater_is_better":     True,

    # --- io ------------------------------------------------------------------
    "train_file":  f"{DATA_ROOT}/train/train.jsonl",
    "val_file":    f"{DATA_ROOT}/val/val.jsonl",
    "test_file":   f"{DATA_ROOT}/test/test.jsonl",
    "output_dir":  f"{WORK}/outputs",
    "adapter_dir": f"{WORK}/networking_orpo_lora",
    "report_to":   "none",      # "wandb" if you want W&B (add the WANDB_API_KEY secret)

    # --- prompt template -----------------------------------------------------
    # Used identically at training and inference time. Every example shares this
    # header, so it does little teaching on its own — its job is to anchor the
    # format and to not contradict the data.
    "prompt_template": (
        "You are a computer networking expert. "
        "Answer the question accurately.\n\n"
        "### Question:\n{}\n\n### Answer:\n"
    ),

    # Alternatives — swap in by replacing the value above:
    #
    # (b) domain-stated, no role:
    #   "Below is a question about computer networking. "
    #   "Write a factually correct response.\n\n"
    #   "### Question:\n{}\n\n### Answer:\n"
    #
    # (c) bare — lowest token overhead, relies entirely on the 2,416 examples:
    #   "### Question:\n{}\n\n### Answer:\n"
}

NUM_GPUS    = torch.cuda.device_count()
CONFIG_PATH = f"{WORK}/orpo_config.json"
json.dump(CFG, open(CONFIG_PATH, "w"), indent=2)

n_train  = sum(1 for _ in open(CFG["train_file"]))
eff      = CFG["per_device_train_batch_size"] * CFG["gradient_accumulation_steps"] * NUM_GPUS
spe      = n_train // eff
print("effective batch size :", eff)
print("steps / epoch        :", spe)
print("max total steps      :", spe * CFG["num_train_epochs"],
      f"({CFG['num_train_epochs']} epochs, before early stopping)")
print(f"rough wall clock     : ~{spe*CFG['num_train_epochs']*7/3600:.1f} h if it runs "
      f"all {CFG['num_train_epochs']} epochs  <-- check against Kaggle's session limit")

effective batch size : 16
steps / epoch        : 151
max total steps      : 4530 (30 epochs, before early stopping)
rough wall clock     : ~8.8 h if it runs all 30 epochs  <-- check against Kaggle's session limit


## 4. Verify the split

Cheap guard, worth keeping: confirms no pair and no question leaks across train / val /
test before a single GPU-hour is spent. It should print all zeros.

In [5]:
import json

def load(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

train, val, test = load(CFG["train_file"]), load(CFG["val_file"]), load(CFG["test_file"])

row    = lambda r: tuple(r[k].strip() for k in ("prompt", "chosen", "rejected"))
prompt = lambda r: r["prompt"].strip()

R = {n: {row(r)    for r in s} for n, s in (("train",train),("val",val),("test",test))}
P = {n: {prompt(r) for r in s} for n, s in (("train",train),("val",val),("test",test))}

total = len(train) + len(val) + len(test)
print(f"train / val / test = {len(train):,} / {len(val):,} / {len(test):,}"
      f"   ({100*len(train)/total:.1f}% / {100*len(val)/total:.1f}% / {100*len(test)/total:.1f}%)")
print()
for a, b in (("train","val"), ("train","test"), ("val","test")):
    print(f"  identical pairs {a:5s} & {b:4s}: {len(R[a] & R[b])}")
    print(f"  shared prompts  {a:5s} & {b:4s}: {len(P[a] & P[b])}")
print()
for n, s in (("train",train), ("val",val), ("test",test)):
    print(f"  duplicate rows within {n:5s}: {len(s) - len(R[n])}")
print(f"  rows with chosen == rejected : "
      f"{sum(r['chosen'].strip() == r['rejected'].strip() for r in train+val+test)}")
print(f"  rows with an empty field     : "
      f"{sum(not all(str(r.get(k,'')).strip() for k in ('prompt','chosen','rejected')) for r in train+val+test)}")

assert not (P["train"] & P["val"]) and not (P["train"] & P["test"]), "train/eval leakage!"
print("\nsplit is clean — safe to train")

train / val / test = 2,416 / 303 / 302   (80.0% / 10.0% / 10.0%)

  identical pairs train & val : 0
  shared prompts  train & val : 0
  identical pairs train & test: 0
  shared prompts  train & test: 0
  identical pairs val   & test: 0
  shared prompts  val   & test: 0

  duplicate rows within train: 0
  duplicate rows within val  : 0
  duplicate rows within test : 0
  rows with chosen == rejected : 0
  rows with an empty field     : 0

split is clean — safe to train


In [6]:
import textwrap
r = train[0]
for k in ("prompt", "chosen", "rejected"):
    print(k.upper(), "=" * (60 - len(k)))
    print(textwrap.fill(r[k], 90))
print("\nFormatted prompt as the model will see it:")
print("-" * 70)
print(CFG["prompt_template"].format(r["prompt"]))

PROMPT ======================================================
What is a client in networking?
CHOSEN ======================================================
A client is a device that accesses a service provided by a server. For example, a laptop
loading a web page from a web server is acting as a client.
REJECTED ====================================================
A client is a low-powered computer that cannot function without being connected to a
server at all times.

Formatted prompt as the model will see it:
----------------------------------------------------------------------
You are a computer networking expert. Answer the question accurately.

### Question:
What is a client in networking?

### Answer:



## 5. Write the training script

`%%writefile` drops this to disk; `torchrun` then runs it once per GPU.

In [7]:
%%writefile /kaggle/working/orpo_train.py
# =============================================================================
# orpo_train.py — DDP ORPO training for Kaggle 2x T4
# Launched with: torchrun --nproc_per_node=2 orpo_train.py
# All knobs come from orpo_config.json, written by the notebook.
# =============================================================================
import os, sys, json, torch

# Must be imported before transformers/trl so Unsloth can patch them.
from unsloth import FastLanguageModel, PatchDPOTrainer, is_bfloat16_supported
PatchDPOTrainer()

from datasets import load_dataset
from trl import ORPOConfig, ORPOTrainer
from transformers import set_seed, EarlyStoppingCallback

LOCAL_RANK = int(os.environ.get("LOCAL_RANK", 0))
WORLD_SIZE = int(os.environ.get("WORLD_SIZE", 1))
IS_MAIN    = LOCAL_RANK == 0

# Every rank runs this whole file, so without this every banner, warning and
# library message appears once per GPU. Send the replicas' stdout to a log file
# and keep stderr, so a crash on rank 1 is still visible.
if not IS_MAIN:
    import warnings, logging
    _replica_log = open(f"/kaggle/working/rank{LOCAL_RANK}.log", "w")
    sys.stdout = _replica_log
    warnings.filterwarnings("ignore")
    logging.getLogger().setLevel(logging.ERROR)
    os.environ["TQDM_DISABLE"] = "1"      # transformers already hides replica bars
    import transformers.utils.logging as _tl, datasets.utils.logging as _dl
    _tl.set_verbosity_error(); _dl.set_verbosity_error(); _dl.disable_progress_bar()

# Each DDP rank must own exactly one GPU, and the model must be loaded onto it.
torch.cuda.set_device(LOCAL_RANK)

CFG = json.load(open(os.environ["ORPO_CONFIG"]))
set_seed(CFG["seed"])

if IS_MAIN:
    print(f"[rank0] world_size={WORLD_SIZE}  visible GPUs={torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"        cuda:{i} = {torch.cuda.get_device_name(i)}")

# ---------------------------------------------------------------- model ----
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = CFG["model_name"],
    max_seq_length = CFG["max_seq_length"],
    dtype          = None,               # auto -> float16 on T4
    load_in_4bit   = CFG["load_in_4bit"],
    device_map     = {"": LOCAL_RANK},   # <-- pin this rank's replica to its own GPU
    # token = os.environ.get("HF_TOKEN"),  # only for gated models
)

model = FastLanguageModel.get_peft_model(
    model,
    r                          = CFG["lora_r"],
    target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj"],
    lora_alpha                 = CFG["lora_alpha"],
    # Unsloth's fast path is the dropout=0 case; a nonzero value works but is
    # slower. Worth it here: small dataset + many epochs = overfitting risk.
    lora_dropout               = CFG["lora_dropout"],
    bias                       = "none",
    use_gradient_checkpointing = CFG["use_gradient_checkpointing"],
    random_state               = CFG["seed"],
    use_rslora                 = False,
    loftq_config               = None,
)

# --------------------------------------------------------------- dataset ---
PROMPT_TEMPLATE = CFG["prompt_template"]
EOS = tokenizer.eos_token

def format_pair(sample):
    return {
        "prompt":   PROMPT_TEMPLATE.format(sample["prompt"].strip()),
        "chosen":   sample["chosen"].strip()   + EOS,
        "rejected": sample["rejected"].strip() + EOS,
    }

raw = load_dataset(
    "json",
    data_files = {"train": CFG["train_file"], "validation": CFG["val_file"]},
)
train_ds = raw["train"].map(format_pair, remove_columns=raw["train"].column_names)
val_ds   = raw["validation"].map(format_pair, remove_columns=raw["validation"].column_names)

if IS_MAIN:
    print(f"[rank0] train={len(train_ds)}  val={len(val_ds)}")
    print("[rank0] --- sample prompt ---")
    print(train_ds[0]["prompt"])
    print("[rank0] --- sample chosen ---")
    print(train_ds[0]["chosen"])

# --------------------------------------------------------------- trainer ---
# "epoch"  -> evaluate + checkpoint once per epoch (3 candidates over 3 epochs)
# "steps"  -> every CFG["eval_steps"] optimiser steps (finer selection)
STRATEGY  = CFG.get("eval_save_strategy", "epoch")
STEP_ARGS = ({"eval_steps": CFG["eval_steps"], "save_steps": CFG["eval_steps"]}
             if STRATEGY == "steps" else {})
if IS_MAIN:
    print(f"[rank0] eval/save strategy = {STRATEGY}"
          + (f" every {CFG['eval_steps']} steps" if STEP_ARGS else " (once per epoch)"))

args = ORPOConfig(
    # --- ORPO ---
    beta                  = CFG["beta"],
    max_length            = CFG["max_seq_length"],
    max_prompt_length     = CFG["max_prompt_length"],
    max_completion_length = CFG["max_seq_length"] - CFG["max_prompt_length"],

    # --- batch / schedule ---
    # Effective batch = per_device * grad_accum * WORLD_SIZE
    per_device_train_batch_size = CFG["per_device_train_batch_size"],
    per_device_eval_batch_size  = CFG["per_device_eval_batch_size"],
    gradient_accumulation_steps = CFG["gradient_accumulation_steps"],
    num_train_epochs            = CFG["num_train_epochs"],
    learning_rate               = CFG["learning_rate"],
    lr_scheduler_type           = "linear",
    warmup_steps                = CFG["warmup_steps"],
    weight_decay                = 0.01,
    optim                       = "adamw_8bit",
    max_grad_norm               = 1.0,

    # --- precision: T4 is Turing, fp16 only (no bf16) ---
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),

    # --- DDP ---
    # LoRA freezes the base weights, so DDP would otherwise complain about
    # parameters that never receive a gradient.
    ddp_find_unused_parameters = False,
    dataloader_num_workers     = 2,
    log_level                  = "warning",
    log_level_replica          = "error",   # replicas stay quiet
    dataset_num_proc           = 1,   # dataset is small; avoids fork storms per rank

    # --- eval / logging / saving ---
    # Eval and save must use the SAME strategy for load_best_model_at_end;
    # in "steps" mode save_steps must also be a multiple of eval_steps.
    # At the end of training the best checkpoint is reloaded into the model,
    # so the adapter saved below is the best one, not the last step's.
    eval_strategy          = STRATEGY,
    save_strategy          = STRATEGY,
    save_total_limit       = CFG["save_total_limit"],
    load_best_model_at_end = True,
    metric_for_best_model  = CFG["metric_for_best_model"],
    greater_is_better      = CFG["greater_is_better"],
    logging_steps          = CFG["logging_steps"],
    output_dir             = CFG["output_dir"],
    report_to       = CFG["report_to"],
    seed            = CFG["seed"],
    **STEP_ARGS,
)

# Stop once the val metric has failed to improve for N consecutive evals, so a
# large num_train_epochs is an upper bound rather than a promise to burn it all.
# Requires load_best_model_at_end + metric_for_best_model, both set above.
callbacks = []
patience  = CFG.get("early_stopping_patience", 0)
if patience:
    callbacks.append(EarlyStoppingCallback(early_stopping_patience=patience))
    if IS_MAIN:
        print(f"[rank0] early stopping: patience = {patience} evals")

trainer = ORPOTrainer(
    model             = model,
    args              = args,
    train_dataset     = train_ds,
    eval_dataset      = val_ds,
    processing_class  = tokenizer,   # NOTE: trl>=0.12 renamed `tokenizer` -> `processing_class`
    callbacks         = callbacks,
)

if IS_MAIN:
    eff = (CFG["per_device_train_batch_size"]
           * CFG["gradient_accumulation_steps"] * WORLD_SIZE)
    print(f"[rank0] effective batch size = {eff}")

stats = trainer.train()

# ------------------------------------------------------------------ save ---
if trainer.is_world_process_zero():
    print(stats)
    print(f"[rank0] best checkpoint : {trainer.state.best_model_checkpoint}")
    print(f"[rank0] best {CFG['metric_for_best_model']} : {trainer.state.best_metric}")
    model.save_pretrained(CFG["adapter_dir"])
    tokenizer.save_pretrained(CFG["adapter_dir"])
    with open(os.path.join(CFG["adapter_dir"], "train_log.json"), "w") as f:
        json.dump(trainer.state.log_history, f, indent=2)
    print(f"[rank0] adapter saved -> {CFG['adapter_dir']}")

if torch.distributed.is_initialized():
    torch.distributed.barrier()
    torch.distributed.destroy_process_group()


Writing /kaggle/working/orpo_train.py


## 6. Launch DDP training on both T4s

`NCCL_P2P_DISABLE` / `NCCL_IB_DISABLE` are the standard fix for the NCCL hang on Kaggle's
T4 pair — they sit on plain PCIe with no peer-to-peer or InfiniBand link.

The progress bar you see is rank 0's; the step counts are already global. Expect roughly
40–70 minutes for 3 epochs.

In [8]:
%%time
import os
os.environ["ORPO_CONFIG"]            = CONFIG_PATH
os.environ["NCCL_P2P_DISABLE"]       = "1"   # Kaggle T4 pair: no P2P -> NCCL would hang
os.environ["NCCL_IB_DISABLE"]        = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"]        = "2"

!cd /kaggle/working && \
  ORPO_CONFIG={CONFIG_PATH} NCCL_P2P_DISABLE=1 NCCL_IB_DISABLE=1 \
  TOKENIZERS_PARALLELISM=false OMP_NUM_THREADS=2 \
  torchrun --nproc_per_node={NUM_GPUS} --master_port=29501 orpo_train.py

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
🦥 Unsloth Zoo will now patch everything to make training faster!
[rank0] world_size=2  visible GPUs=2
        cuda:0 = Tesla T4
        cuda:1 = Tesla T4
==((====))==  Unsloth 2026.9.12: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth 2026.9.12 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.
Map: 100%|██████████████████████████| 303/303 [00:00<00:00, 14632.14 examples/s]
[rank0] train=241

## 7. Reload the adapter and generate

Training ran in a separate process, so the notebook kernel loads the saved adapter fresh —
on a single GPU, which is all inference needs.

In [9]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = CFG["adapter_dir"],
    max_seq_length = CFG["max_seq_length"],
    dtype          = None,
    load_in_4bit   = CFG["load_in_4bit"],
)
FastLanguageModel.for_inference(model)

def ask(question, max_new_tokens=128):
    inputs = tokenizer([CFG["prompt_template"].format(question)],
                       return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         use_cache=True, do_sample=False)
    text = tokenizer.batch_decode(out, skip_special_tokens=True)[0]
    return text.split("### Answer:")[-1].strip()


print(ask("What is the difference between a collision domain and a broadcast domain?"))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.12: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
total weights      : 4.473 GiB
no_split classes   : ['LlamaDecoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.342 GiB
activation reserve : 10.570 GiB requested
  cuda:0  budget  12.98 GiB  weights  2.378 GiB  free 10.599 GiB  reserve 10.570 GiB
  cuda:1  budget  12.98 GiB  weights  2.096 GiB  free 10.883 GiB  reserve 10.400 GiB   <- output head
note: Bnb4BitHfQuantizer.adjust_max_mem

Unsloth 2026.9.12 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


A collision domain is a group of devices that share the same physical network segment. A broadcast domain is a group of devices that receive broadcast frames sent by any member.


In [10]:
import textwrap
for r in test[:5]:
    print("Q:", r["prompt"])
    print("  model    :", textwrap.fill(ask(r["prompt"]), 90, subsequent_indent=" " * 13))
    print("  reference:", textwrap.fill(r["chosen"], 90, subsequent_indent=" " * 13))
    print()

Q: What is the difference between a host-based firewall and a network firewall?
  model    : A host-based firewall is on the device itself, while a network firewall is a separate
             device that protects an entire network.
  reference: A host-based firewall runs on an individual device and controls that device's traffic. A
             network firewall is a separate hardware appliance that controls traffic for
             the entire network.

Q: What is a network firewall?
  model    : A device that filters traffic between networks, typically between the internet and a
             private network.
  reference: A network firewall is a hardware appliance that inspects all traffic entering and exiting
             a network and allows or denies it based on configured rules, protecting the
             network from malicious traffic.

Q: Where is copper UTP typically used in a network?
  model    : In the access layer, connecting end hosts to switches.
  reference: Mostly for co

## 8. Preference accuracy on the held-out test set

The direct question for an ORPO run: does the model now assign higher likelihood to the
chosen answer than to the rejected one, on 302 questions it has never seen? Length-normalised
so short answers aren't favoured. A base model sits near 50–60%; a good run lands well above.

In [11]:
import torch
from tqdm.auto import tqdm

@torch.no_grad()
def avg_logprob(prompt, answer):
    p_ids = tokenizer(CFG["prompt_template"].format(prompt),
                      return_tensors="pt").input_ids.to("cuda")
    a_ids = tokenizer(answer + tokenizer.eos_token, add_special_tokens=False,
                      return_tensors="pt").input_ids.to("cuda")
    ids    = torch.cat([p_ids, a_ids], dim=1)
    logits = model(ids).logits[:, :-1].float().log_softmax(-1)
    tgt    = ids[:, 1:]
    lp     = logits.gather(-1, tgt.unsqueeze(-1)).squeeze(-1)[:, p_ids.shape[1] - 1:]
    return lp.mean().item()

wins = sum(avg_logprob(r["prompt"], r["chosen"]) > avg_logprob(r["prompt"], r["rejected"])
           for r in tqdm(test, desc="scoring"))
print(f"\npreference accuracy: {wins}/{len(test)} = {100*wins/len(test):.1f}%")

scoring:   0%|          | 0/302 [00:00<?, ?it/s]


preference accuracy: 233/302 = 77.2%


## 9. Save / export

In [12]:
ADAPTER = CFG["adapter_dir"]

# The LoRA adapter is already saved there by the training script.
# Uncomment whichever export you need.

# --- push adapter to the Hub -------------------------------------------------
# model.push_to_hub("YOUR_USERNAME/llama3-8b-networking-orpo", token="YOUR_HF_TOKEN")
# tokenizer.push_to_hub("YOUR_USERNAME/llama3-8b-networking-orpo", token="YOUR_HF_TOKEN")

# --- merge to 16-bit (needs ~16 GB disk) -------------------------------------
# model.save_pretrained_merged("/kaggle/working/networking_orpo_16bit", tokenizer,
#                              save_method="merged_16bit")

# --- GGUF for llama.cpp / Ollama ---------------------------------------------
# model.save_pretrained_gguf("/kaggle/working/networking_orpo_gguf", tokenizer,
#                            quantization_method="q4_k_m")

!du -sh {ADAPTER}
!ls -la {ADAPTER}

177M	/kaggle/working/networking_orpo_lora
total 180904
drwxr-xr-x 2 root root      4096 Sep 30 12:18 .
drwxr-xr-x 6 root root      4096 Sep 30 12:18 ..
-rw-r--r-- 1 root root      1240 Sep 30 12:18 adapter_config.json
-rw-r--r-- 1 root root 167832240 Sep 30 12:18 adapter_model.safetensors
-rw-r--r-- 1 root root      5231 Sep 30 12:18 README.md
-rw-r--r-- 1 root root       464 Sep 30 12:18 special_tokens_map.json
-rw-r--r-- 1 root root     50644 Sep 30 12:18 tokenizer_config.json
-rw-r--r-- 1 root root  17209961 Sep 30 12:18 tokenizer.json
-rw-r--r-- 1 root root    117191 Sep 30 12:18 train_log.json
